In [1]:
from composer import (
    Agent,
    Vector,
    MCPClient,
    combine_tools,
    Thread,
    SystemMessage,
    HumanMessage,
    ImageMessage,
    ThinkingEvent,
    ToolCallEvent,
    ToolResultEvent,
    AssistantEvent,
    ToolResultHideRule
)
import subprocess

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
mcp = MCPClient(
    servers={
        "task_manager": {
            "transport": "http",
            "url": "http://127.0.0.1:8000/mcp",
            # optional:
            # "headers": {"Authorization": "Bearer ..."},
            # "timeout": 30,  # seconds (see langchain-mcp-adapters docs)
        },
        "butcher": {
            "transport": "http",
            "url": "http://127.0.0.1:3333/mcp",
        },
    },
    tool_name_prefix=True,  # tools become task_manager_<name> if you add more servers
)

In [4]:
from langchain_core.tools import tool

@tool
def run_terminal_command(command: str) -> str:
    """Safely executes a shell command in a subprocess and returns stdout/stderr."""
    try:
        # Run command securely without shell=True to avoid injection issues
        result = subprocess.run(
            command.split(),
            capture_output=True,
            text=True,
            timeout=15
        )
        if result.returncode == 0:
            return f"Success:\n{result.stdout}"
        else:
            return f"Error (Exit Code {result.returncode}):\n{result.stderr}"
    except Exception as e:
        return f"Execution Failed: {str(e)}"

In [5]:
tools = await mcp.load_tools()

In [6]:
await mcp.load_resources()
blobs = await mcp.get_resource("taskmanager://server-info")
server_info = blobs[0].as_string()

In [7]:
await mcp.load_prompts()
prompts = await mcp.get_prompt("agent_system_prompt", server="task_manager")
task_manager_system_prompt = prompts[0].content
prompts = await mcp.get_prompt("agent_system_prompt", server="butcher")
butcher_system_prompt = prompts[0].content

In [8]:
model="openai/gpt-oss-120b:free"
emb_model="nvidia/llama-nemotron-embed-vl-1b-v2:free"
vision="sourceful/riverflow-v2.5-fast:free"

In [9]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model=model,
    api_key=os.getenv("API_KEY"),
    temperature=0,
    max_tokens=4096,
    reasoning={"effort": "medium", "summary": "auto"},
)

agent = Agent(
    model=llm,
    tools=tools,
)

In [10]:
# agent = Agent(
#     model=model,
#     base_url=os.getenv("BASE_URL"),
#     api_key=os.getenv("API_KEY"),
#     tools=tools,
#     reasoning={"effort": "medium"},  # or reasoning=True
# )

In [11]:
thread = Thread()

In [12]:
system = f"""
You are an ReAct and conversational agent. your goal is to perform and complete the user provided task.
- you will be having access to multiple MCP server tool which you can use to complete the user provided task and it's objective.
- on completion of the task always respond to user in well defined report.
- for conversational based query as per the query respond in general conversation increment way not like the report based.
- always mention the task id if made task and used task manager server in final response report. 

---
{butcher_system_prompt}
---
{task_manager_system_prompt}
"""

In [13]:
SystemMessage(system) | thread

Thread(messages=1)

In [14]:
thread.append(HumanMessage("hi"))

In [15]:
(await (thread | agent)).content

'Hello! How can I help you today?'

In [16]:
# HumanMessage("I want you to go to https://practice.expandtesting.com/upload upload using text+filename method with filename test.txt with text `testing` and upload it with capturing request and register it as well") | thread

In [24]:
HumanMessage("I want you to goto https://rac.gov.in/drdo/public/login and analyse the webpage/instruction etc. fill out the form with any values and captcha value capture the request and register request, in case of credential error just stop the tsk there") | thread

Thread(messages=44)

In [25]:
# HumanMessage("go to https://2captcha.com/demo/normal and read the assignment and do it") | thread

In [50]:
HumanMessage("okay bye, thanks") | thread

Thread(messages=120)

In [51]:
thread.add_tool_hide_rule(ToolResultHideRule(server="butcher", tool_name="interactables", on_hide_message="[this tool result was colapsed please refer to the latest one only]"))
thread.add_tool_hide_rule(ToolResultHideRule(server="butcher", tool_name="snapshot", on_hide_message="[this tool result was colapsed please refer to the latest one only]"))

In [52]:
in_thinking = False
in_assistant = False

for event in agent.stream_events(thread):
    if isinstance(event, ThinkingEvent):
        if not in_thinking:
            print("[think] ", end="", flush=True)
            in_thinking = True
        print(event.text, end="", flush=True)

    elif isinstance(event, AssistantEvent):
        if in_thinking:
            print("\n\n[Response]\n", end="")  # blank line after thinking
            in_thinking = False
        if not in_assistant:
            in_assistant = True
        print(event.text, end="", flush=True)

    elif isinstance(event, ToolCallEvent):
        if in_thinking:
            print("\n", end="")
            in_thinking = False
        print(f"\n[tool] {event.call.name}", flush=True)

print()  # final newline

You’re welcome! If you need anything else later, just let me know. Have a great day!


In [48]:
thread.token_count()

39072

In [28]:
v = Vector(
    model=emb_model,
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
)

In [29]:
e = v.vector("hii")

In [30]:
e.shape

(2048,)

In [31]:
type(e)

numpy.ndarray

In [40]:
import numpy as np

In [43]:
e[:3].astype(np.float64)

array([-0.02453613,  0.02566528,  0.03222656])

In [42]:
e[:3].astype(np.float32)

array([-0.02453613,  0.02566528,  0.03222656], dtype=float32)

In [41]:
e[:3].astype(np.float16)

array([-0.02454,  0.02567,  0.03223], dtype=float16)